# Pandas高级数据处理

In [2]:
import numpy as np
import pandas as pd

## 级联

级联（Concatenation）是指将多个数据结构（如DataFrame或Series）沿着某个轴进行拼接。Pandas提供了`pd.concat()`函数来实现级联操作。

- 级联语法的核心是索引对齐。
- 级联的应用场景：不同期，但是结构相同的数据汇总。

In [5]:
df1 = pd.DataFrame(data = [[1, 2, 3, 4]], columns=list('ABCD'))
df2 = pd.DataFrame(data = [[5, 6, 7, 8]], columns=list('ABCD'))
display(df1, df2)

,A,B,C,D
0,1,2,3,4


,A,B,C,D
0,5,6,7,8


In [9]:
res1 = pd.concat([df1, df2], axis=0)  # 按行拼接,拼接后有重复索引
res1

,A,B,C,D
0,1,2,3,4
0,5,6,7,8


In [10]:
res2 = pd.concat([df1, df2], axis=1)  # 按列拼接，拼接后有重复列名
res2

,A,B,C,D,A,B,C,D
0,1,2,3,4,5,6,7,8


In [11]:
res1.loc[0]  # 按行索引选取数据，返回的是一个Series对象

,A,B,C,D
0,1,2,3,4
0,5,6,7,8


In [14]:
res2.loc[:,'A']  # 按列索引选取数据，返回的是一个Series对象

,A,A
0,1,5


In [16]:
# pd.concat([df1, df2], axis=0,verify_integrity=True)  # 按行拼接,拼接后有重复索引，抛出异常

In [19]:
# pd.concat([df1, df2], axis=1,verify_integrity=True)  # 按列拼接,拼接后有重复列名,抛出异常

> 解决索引重复问题：改为多层索引   
上半年&emsp;&emsp;A  
&emsp;&emsp;&emsp;&emsp;&emsp;B  
&emsp;&emsp;&emsp;&emsp;&emsp;C  
&emsp;&emsp;&emsp;&emsp;&emsp;D  
下半年&emsp;&emsp;A  
&emsp;&emsp;&emsp;&emsp;&emsp;B  
&emsp;&emsp;&emsp;&emsp;&emsp;C  
&emsp;&emsp;&emsp;&emsp;&emsp;D  

In [22]:
# 通过多层索引解决索引重复问题
pd.concat([df1, df2], axis=0, keys=['上半年', '下半年'])  # 按行拼接,在索引上增加一层索引，解决索引重复问题

,,A,B,C,D
上半年,0,1,2,3,4
下半年,0,5,6,7,8


In [23]:
# 解决索引重复问题：改为多层索引
pd.concat([df1, df2], axis=1, keys=['上半年', '下半年'])  # 按行拼接,在索引上增加一层索引，解决索引重复问题

上半年          下半年         
    A  B  C  D   A  B  C  D
0   1  2  3  4   5  6  7  8

In [26]:
# 使用names参数为多层索引命名
pd.concat([df1, df2], axis=1, keys=['上半年', '下半年'], names=['时间', '产品'])  # 按列拼接,在索引上增加一层索引，解决索引重复问题，并为多层索引命名

时间 上半年          下半年         
产品   A  B  C  D   A  B  C  D
0    1  2  3  4   5  6  7  8

In [27]:
# 使用ignore_index参数忽略索引
pd.concat([df1, df2], axis=0, ignore_index=True)  # 按行拼接,忽略索引，重新生成索引

,A,B,C,D
0,1,2,3,4
1,5,6,7,8


In [32]:
df3 = pd.DataFrame(data = [[1, 2, 3, 4, 5]], columns=list('CDEAB'))
df3

,C,D,E,A,B
0,1,2,3,4,5


In [33]:
# 不同形状的DataFrame拼接,columns不一致，按行拼接时，缺失的列会被补充为NaN
pd.concat([df1, df3], axis=0, ignore_index=True)  # 按行拼接,忽略索引，重新生成索引

,A,B,C,D,E
0,1,2,3,4,NaN
1,4,5,1,2,3.0


In [36]:
df4 = pd.DataFrame(data = np.random.randint(1, 10, size=(3, 4)), columns=list('ABCD'))
df5 = pd.DataFrame(data = np.random.randint(-10, 1, size=(3, 4)), columns=list('BCDE'))
display(df4, df5)

,A,B,C,D
0,8,6,2,2
1,9,8,7,6
2,2,4,9,6


,B,C,D,E
0,-6,0,-10,-8
1,0,-10,-10,-5
2,-8,-1,-2,-10


In [38]:
pd.concat([df4, df5], axis=0, ignore_index=True)  # 按行拼接,忽略索引，重新生成索引

,A,B,C,D,E
0,8.0,6,2,2,NaN
1,9.0,8,7,6,NaN
2,2.0,4,9,6,NaN
3,NaN,-6,0,-10,-8.0
4,NaN,0,-10,-10,-5.0
5,NaN,-8,-1,-2,-10.0


In [40]:
# outer join:取并集，保留级联索引中所有的标签
# inner join:取交集，保留级联索引中共有的标签
pd.concat([df4, df5], axis=0, ignore_index=True,join='inner')  # 按行拼接,忽略索引，重新生成索引

,B,C,D
0,6,2,2
1,8,7,6
2,4,9,6
3,-6,0,-10
4,0,-10,-10
5,-8,-1,-2


## 合并

合并就是根据两张表的公共信息，把两张表的数据汇总的方法。  
合并以列的内容为参考标准，不存在行合并，都是列合并。   
合并的列通常是离散型数据（如ID、姓名等）。    
合并的列之间存在一对一、一对多、多对一、多对多的关系。

In [42]:
# 读取合并表格案例.excel文件
first_half_year = pd.read_excel('合并表格案例.xlsx', sheet_name=0)
second_half_year = pd.read_excel('合并表格案例.xlsx', sheet_name=1)

In [43]:
user_table = pd.read_excel('合并表格案例.xlsx', sheet_name=2)
product_table = pd.read_excel('合并表格案例.xlsx', sheet_name=3)
return_table = pd.read_excel('合并表格案例.xlsx', sheet_name=4)

In [44]:
display(first_half_year, second_half_year, user_table, product_table, return_table)

,用户ID,商品ID,订单ID,购买数量
0,U001,P001,H2026001,1
1,U002,P003,H2026002,2
2,U003,P006,H2026003,3
3,U004,P004,H2026004,1
4,U005,P002,H2026005,2
5,U006,P007,H2026006,1
6,U007,P003,H2026007,4
7,U008,P005,H2026008,1
8,U009,P008,H2026009,1
9,U010,P006,H2026010,5


,用户ID,商品ID,订单ID,购买数量
0,U003,P002,L2026001,2
1,U001,P007,L2026002,1
2,U006,P003,L2026003,3
3,U008,P004,L2026004,2
4,U002,P005,L2026005,1
5,U011,P008,L2026006,1
6,U005,P006,L2026007,4
7,U009,P001,L2026008,2
8,U004,P003,L2026009,2
9,U007,P002,L2026010,3


,用户ID,地区,VIP等级,手机号
0,U001,北京市,铂金会员,13800001001
1,U002,上海市,黄金会员,13900001002
2,U003,广东省,白银会员,13700001003
3,U004,浙江省,黄金会员,13600001004
4,U005,江苏省,普通会员,13500001005
5,U006,四川省,白银会员,13400001006
6,U007,湖北省,普通会员,13100001007
7,U008,山东省,黄金会员,13200001008
8,U009,福建省,白银会员,13300001009
9,U010,河南省,普通会员,15000001010


,商品ID,商品类别,商品品牌,商品单价
0,P001,数码配件,绿联,129.0
1,P002,家居日用,洁丽雅,39.9
2,P003,休闲食品,三只松鼠,59.8
3,P004,服饰,优衣库,199.0
4,P005,美妆护肤,珀莱雅,269.0
5,P006,办公用品,得力,25.5
6,P007,小家电,美的,399.0
7,P008,运动户外,李宁,329.0


,订单_ID,退货状态
0,H2026002,已退货
1,H2026006,退货中
2,H2026009,退货驳回
3,L2026004,已退货
4,L2026008,退货中
5,L2026011,已退货


### 1.计算上半年订单总额

In [49]:
# 两张表合并时，默认是根据所有的相同字段名称的列进行合并的，若两张表中有相同的列名，则会根据这些列名进行合并。
res1 = pd.merge(left=first_half_year, right=product_table) 
res1

,用户ID,商品ID,订单ID,购买数量,商品类别,商品品牌,商品单价
0,U001,P001,H2026001,1,数码配件,绿联,129.0
1,U002,P003,H2026002,2,休闲食品,三只松鼠,59.8
2,U003,P006,H2026003,3,办公用品,得力,25.5
3,U004,P004,H2026004,1,服饰,优衣库,199.0
4,U005,P002,H2026005,2,家居日用,洁丽雅,39.9
5,U006,P007,H2026006,1,小家电,美的,399.0
6,U007,P003,H2026007,4,休闲食品,三只松鼠,59.8
7,U008,P005,H2026008,1,美妆护肤,珀莱雅,269.0
8,U009,P008,H2026009,1,运动户外,李宁,329.0
9,U010,P006,H2026010,5,办公用品,得力,25.5


In [54]:
res1['购买数量'] * res1['商品单价']  # 计算订单总额

0     129.0
1     119.6
2      76.5
3     199.0
4      79.8
5     399.0
6     239.2
7     269.0
8     329.0
9     127.5
10    258.0
dtype: float64

In [55]:
(res1['购买数量'] * res1['商品单价']).sum()  # 计算订单总额

np.float64(2225.6)

### 2.获取上半年用户地区，查看各地区订单数量

In [62]:
#how='left'：左连接，保留左表的所有数据，右表中没有匹配的数据用NaN填充
#how='right'：右连接，保留右表的所有数据，左表中没有匹配的数据用NaN填充
#how='outer'：外连接，保留左表和右表的所有数据，没有匹配的数据用NaN填充
#how='inner'：内连接，只保留左表和右表中都有的数据
#how='cross'：笛卡尔积连接，返回左表和右表的所有组合
res2 = pd.merge(left=first_half_year, right=user_table,how='inner')  # 内连接，只保留左表和右表中都有的数据

In [64]:
res2

,用户ID,商品ID,订单ID,购买数量,地区,VIP等级,手机号
0,U001,P001,H2026001,1,北京市,铂金会员,13800001001
1,U002,P003,H2026002,2,上海市,黄金会员,13900001002
2,U003,P006,H2026003,3,广东省,白银会员,13700001003
3,U004,P004,H2026004,1,浙江省,黄金会员,13600001004
4,U005,P002,H2026005,2,江苏省,普通会员,13500001005
5,U006,P007,H2026006,1,四川省,白银会员,13400001006
6,U007,P003,H2026007,4,湖北省,普通会员,13100001007
7,U008,P005,H2026008,1,山东省,黄金会员,13200001008
8,U009,P008,H2026009,1,福建省,白银会员,13300001009
9,U010,P006,H2026010,5,河南省,普通会员,15000001010


In [66]:
pd.merge(left=first_half_year, right=user_table,how='left')  # 左连接，保留左表的所有数据，右表中没有匹配的数据用NaN填充

,用户ID,商品ID,订单ID,购买数量,地区,VIP等级,手机号
0,U001,P001,H2026001,1,北京市,铂金会员,13800001001
1,U002,P003,H2026002,2,上海市,黄金会员,13900001002
2,U003,P006,H2026003,3,广东省,白银会员,13700001003
3,U004,P004,H2026004,1,浙江省,黄金会员,13600001004
4,U005,P002,H2026005,2,江苏省,普通会员,13500001005
5,U006,P007,H2026006,1,四川省,白银会员,13400001006
6,U007,P003,H2026007,4,湖北省,普通会员,13100001007
7,U008,P005,H2026008,1,山东省,黄金会员,13200001008
8,U009,P008,H2026009,1,福建省,白银会员,13300001009
9,U010,P006,H2026010,5,河南省,普通会员,15000001010


In [67]:
pd.merge(left=first_half_year, right=user_table,how='right')  # 右连接，保留右表的所有数据，左表中没有匹配的数据用NaN填充

,用户ID,商品ID,订单ID,购买数量,地区,VIP等级,手机号
0,U001,P001,H2026001,1,北京市,铂金会员,13800001001
1,U002,P003,H2026002,2,上海市,黄金会员,13900001002
2,U003,P006,H2026003,3,广东省,白银会员,13700001003
3,U004,P004,H2026004,1,浙江省,黄金会员,13600001004
4,U005,P002,H2026005,2,江苏省,普通会员,13500001005
5,U006,P007,H2026006,1,四川省,白银会员,13400001006
6,U007,P003,H2026007,4,湖北省,普通会员,13100001007
7,U008,P005,H2026008,1,山东省,黄金会员,13200001008
8,U009,P008,H2026009,1,福建省,白银会员,13300001009
9,U010,P006,H2026010,5,河南省,普通会员,15000001010


In [68]:
pd.merge(left=first_half_year, right=user_table,how = 'outer')  # 外连接，保留左表和右表的所有数据，没有匹配的数据用NaN填充

,用户ID,商品ID,订单ID,购买数量,地区,VIP等级,手机号
0,U001,P001,H2026001,1,北京市,铂金会员,13800001001
1,U002,P003,H2026002,2,上海市,黄金会员,13900001002
2,U003,P006,H2026003,3,广东省,白银会员,13700001003
3,U004,P004,H2026004,1,浙江省,黄金会员,13600001004
4,U005,P002,H2026005,2,江苏省,普通会员,13500001005
5,U006,P007,H2026006,1,四川省,白银会员,13400001006
6,U007,P003,H2026007,4,湖北省,普通会员,13100001007
7,U008,P005,H2026008,1,山东省,黄金会员,13200001008
8,U009,P008,H2026009,1,福建省,白银会员,13300001009
9,U010,P006,H2026010,5,河南省,普通会员,15000001010


In [69]:
pd.merge(left=first_half_year, right=user_table,how = 'cross')  # 笛卡尔积连接，返回左表和右表的所有组合

,用户ID_x,商品ID,订单ID,购买数量,用户ID_y,地区,VIP等级,手机号
0,U001,P001,H2026001,1,U001,北京市,铂金会员,13800001001
1,U001,P001,H2026001,1,U002,上海市,黄金会员,13900001002
2,U001,P001,H2026001,1,U003,广东省,白银会员,13700001003
3,U001,P001,H2026001,1,U004,浙江省,黄金会员,13600001004
4,U001,P001,H2026001,1,U005,江苏省,普通会员,13500001005
...,...,...,...,...,...,...,...,...
116,U011,P001,H2026011,2,U007,湖北省,普通会员,13100001007
117,U011,P001,H2026011,2,U008,山东省,黄金会员,13200001008
118,U011,P001,H2026011,2,U009,福建省,白银会员,13300001009
119,U011,P001,H2026011,2,U010,河南省,普通会员,15000001010


In [70]:
res2['地区']

0     北京市
1     上海市
2     广东省
3     浙江省
4     江苏省
5     四川省
6     湖北省
7     山东省
8     福建省
9     河南省
10    陕西省
Name: 地区, dtype: str

In [72]:
res2['地区'].value_counts()  # 查看各地区订单   数量

地区
北京市    1
上海市    1
广东省    1
浙江省    1
江苏省    1
四川省    1
湖北省    1
山东省    1
福建省    1
河南省    1
陕西省    1
Name: count, dtype: int64

### 3.找出上半年和下半年购买过相同商品的用户

In [74]:
display(first_half_year, second_half_year)

,用户ID,商品ID,订单ID,购买数量
0,U001,P001,H2026001,1
1,U002,P003,H2026002,2
2,U003,P006,H2026003,3
3,U004,P004,H2026004,1
4,U005,P002,H2026005,2
5,U006,P007,H2026006,1
6,U007,P003,H2026007,4
7,U008,P005,H2026008,1
8,U009,P008,H2026009,1
9,U010,P006,H2026010,5


,用户ID,商品ID,订单ID,购买数量
0,U003,P002,L2026001,2
1,U001,P007,L2026002,1
2,U006,P003,L2026003,3
3,U008,P004,L2026004,2
4,U002,P005,L2026005,1
5,U011,P008,L2026006,1
6,U005,P006,L2026007,4
7,U009,P001,L2026008,2
8,U004,P003,L2026009,2
9,U007,P002,L2026010,3


In [78]:
# on参数指定连接的列名，若两个表中有多个相同字段，使用on来指定连接的列名
pd.merge(left=first_half_year, right=second_half_year,on=['用户ID','商品ID']) 

,用户ID,商品ID,订单ID_x,购买数量_x,订单ID_y,购买数量_y


In [79]:
# suffixes参数为其他相同字段的列名添加后缀，避免列名重复
pd.merge(left=first_half_year, right=second_half_year,on=['用户ID','商品ID'], suffixes=('_上半年', '_下半年'))  # suffixes参数指定连接

,用户ID,商品ID,订单ID_上半年,购买数量_上半年,订单ID_下半年,购买数量_下半年


### 4.查看上半年退货商品的总额

In [83]:
# left_on:指定左表的连接列名，right_on指定右表的连接列名
res1 = pd.merge(left=first_half_year, right=return_table,left_on='订单ID', right_on='订单_ID')  # left_on和right_on参数指定连接的列名，若两个表中有多个相同字段，使用left_on和right_on来指定连接的列名 

In [84]:
res1

,用户ID,商品ID,订单ID,购买数量,订单_ID,退货状态
0,U002,P003,H2026002,2,H2026002,已退货
1,U006,P007,H2026006,1,H2026006,退货中
2,U009,P008,H2026009,1,H2026009,退货驳回


In [88]:
res2 = pd.merge(left=res1, right=product_table)

In [89]:
res2

,用户ID,商品ID,订单ID,购买数量,订单_ID,退货状态,商品类别,商品品牌,商品单价
0,U002,P003,H2026002,2,H2026002,已退货,休闲食品,三只松鼠,59.8
1,U006,P007,H2026006,1,H2026006,退货中,小家电,美的,399.0
2,U009,P008,H2026009,1,H2026009,退货驳回,运动户外,李宁,329.0


In [91]:
#过滤掉退货状态为：退货驳回
res3 = res2[res2['退货状态'] != '退货驳回']
res3

,用户ID,商品ID,订单ID,购买数量,订单_ID,退货状态,商品类别,商品品牌,商品单价
0,U002,P003,H2026002,2,H2026002,已退货,休闲食品,三只松鼠,59.8
1,U006,P007,H2026006,1,H2026006,退货中,小家电,美的,399.0


In [92]:
res3['商品单价'] * res3['购买数量']  # 计算退货商品的总额

0    119.6
1    399.0
dtype: float64

In [94]:
(res3['商品单价'] * res3['购买数量']).sum()  # 计算退货商品的总额

np.float64(518.6)

### 5.汇总全年订单数据

In [98]:
total = pd.concat([first_half_year, second_half_year], axis=0, ignore_index=True)  # 按行拼接,忽略索引，重新生成索引

In [99]:
total

,用户ID,商品ID,订单ID,购买数量
0,U001,P001,H2026001,1
1,U002,P003,H2026002,2
2,U003,P006,H2026003,3
3,U004,P004,H2026004,1
4,U005,P002,H2026005,2
5,U006,P007,H2026006,1
6,U007,P003,H2026007,4
7,U008,P005,H2026008,1
8,U009,P008,H2026009,1
9,U010,P006,H2026010,5


### 6.计算全年客单价

In [102]:
res4 = pd.merge(left=total, right=product_table, how='left', on='商品ID')  # 按商品ID进行左连接，保留total表的所有数据，product_table中没有匹配的数据用NaN填充

In [103]:
res4

,用户ID,商品ID,订单ID,购买数量,商品类别,商品品牌,商品单价
0,U001,P001,H2026001,1,数码配件,绿联,129.0
1,U002,P003,H2026002,2,休闲食品,三只松鼠,59.8
2,U003,P006,H2026003,3,办公用品,得力,25.5
3,U004,P004,H2026004,1,服饰,优衣库,199.0
4,U005,P002,H2026005,2,家居日用,洁丽雅,39.9
5,U006,P007,H2026006,1,小家电,美的,399.0
6,U007,P003,H2026007,4,休闲食品,三只松鼠,59.8
7,U008,P005,H2026008,1,美妆护肤,珀莱雅,269.0
8,U009,P008,H2026009,1,运动户外,李宁,329.0
9,U010,P006,H2026010,5,办公用品,得力,25.5


In [104]:
# 增加一列，用来表示每个顾客的消费金额
res4['消费金额'] = res4['商品单价'] * res4['购买数量'] 

In [105]:
res4

,用户ID,商品ID,订单ID,购买数量,商品类别,商品品牌,商品单价,消费金额
0,U001,P001,H2026001,1,数码配件,绿联,129.0,129.0
1,U002,P003,H2026002,2,休闲食品,三只松鼠,59.8,119.6
2,U003,P006,H2026003,3,办公用品,得力,25.5,76.5
3,U004,P004,H2026004,1,服饰,优衣库,199.0,199.0
4,U005,P002,H2026005,2,家居日用,洁丽雅,39.9,79.8
5,U006,P007,H2026006,1,小家电,美的,399.0,399.0
6,U007,P003,H2026007,4,休闲食品,三只松鼠,59.8,239.2
7,U008,P005,H2026008,1,美妆护肤,珀莱雅,269.0,269.0
8,U009,P008,H2026009,1,运动户外,李宁,329.0,329.0
9,U010,P006,H2026010,5,办公用品,得力,25.5,127.5


In [106]:
# 求平均值
res4['消费金额'].mean()  # 计算全年客单价

np.float64(220.74347826086958)

## 分组

分组必聚合

In [109]:
df = pd.read_excel('分组表格案例.xlsx')

In [110]:
df

,菜品,颜色,价格,数量
0,白菜,绿,64,88
1,白菜,红,24,98
2,冬瓜,红,0,43
3,辣椒,红,59,51
4,西红柿,白,29,29
5,白菜,白,5,37
6,冬瓜,红,41,36
7,冬瓜,白,43,24
8,冬瓜,白,88,8
9,辣椒,红,61,28


groupby()用于对数据进行分组操作，返回一个GroupBy对象。

In [113]:
df.groupby(by = '菜品') 

groups()用于查看分组的结果，返回一个字典，key为分组的列名，value为对应的分组数据。

In [114]:
df.groupby(by = '菜品').groups  # groups()用于查看分组的结果，返回一个字典，key为分组的列名，value为对应的分组数据。  

{'冬瓜': [2, 6, 7, 8, 13, 17], '白菜': [0, 1, 5, 12, 16], '西红柿': [4, 10, 15, 19], '辣椒': [3, 9, 11, 14, 18]}

In [117]:
df.loc[[2, 6, 7, 8, 13, 17]]

,菜品,颜色,价格,数量
2,冬瓜,红,0,43
6,冬瓜,红,41,36
7,冬瓜,白,43,24
8,冬瓜,白,88,8
13,冬瓜,绿,18,55
17,冬瓜,白,27,19


In [122]:
# dataframegroupby对象可以直接响应聚合函数
df.groupby(by = '菜品')['数量'].sum()  # 按菜品分组，计算每个菜品的总数量

菜品
冬瓜     185
白菜     361
西红柿    151
辣椒     171
Name: 数量, dtype: int64

In [123]:
df.groupby(by = '菜品')['价格'].mean()  # 按菜品分组，计算每个菜品的平均价格

菜品
冬瓜     36.166667
白菜     28.200000
西红柿    24.500000
辣椒     55.800000
Name: 价格, dtype: float64

多分组

In [129]:
df.groupby(by = ['菜品', '颜色'])

In [130]:
df.groupby(by = ['菜品', '颜色']).groups

{('冬瓜', '白'): Index([7, 8, 17], dtype='int64'),
 ('冬瓜', '红'): RangeIndex(start=2, stop=10, step=4),
 ('冬瓜', '绿'): RangeIndex(start=13, stop=14, step=1),
 ('白菜', '白'): RangeIndex(start=5, stop=27, step=11),
 ('白菜', '红'): RangeIndex(start=1, stop=2, step=1),
 ('白菜', '绿'): RangeIndex(start=0, stop=24, step=12),
 ('西红柿', '白'): RangeIndex(start=4, stop=34, step=15),
 ('西红柿', '红'): RangeIndex(start=10, stop=20, step=5),
 ('辣椒', '红'): Index([3, 9, 11, 18], dtype='int64'),
 ('辣椒', '绿'): RangeIndex(start=14, stop=15, step=1)}

定制多种聚合指标

In [124]:
pgobj = df.groupby(by = '菜品')

In [126]:
# agg 接收一个字典对象
# 字典的key为需要聚合的列名，value为聚合函数名
pgobj.agg({'数量':'sum', '价格':'mean'})  # 定制多种聚合指标，返回一个DataFrame对象

,数量,价格
菜品,,
冬瓜,185,36.166667
白菜,361,28.200000
西红柿,151,24.500000
辣椒,171,55.800000


高级聚合

apply（）函数可以对每个分组应用一个自定义的函数，返回一个新的DataFrame对象。

In [132]:
df.groupby(by = '菜品')[['价格']].apply(np.mean)  # apply()方法可以对分组后的数据进行自定义的聚合操作，返回一个DataFrame对象

菜品
冬瓜     36.166667
白菜     28.200000
西红柿    24.500000
辣椒     55.800000
dtype: float64

In [138]:
def myfunc(x):
    # 对于分组对象来说，apply()方法会将每个分组的数据传递给自定义的函数，函数的参数x就是每个分组的数据
    return x.mean()  # 自定义聚合函数，计算每个分组的平均值

In [139]:
df.groupby(by = '菜品')[['价格']].apply(myfunc)  # apply()方法可以对分组后的数据进行自定义的聚合操作，返回一个DataFrame对象

,价格
菜品,
冬瓜,36.166667
白菜,28.200000
西红柿,24.500000
辣椒,55.800000


## 交叉表

交叉表统计的是两个或多个离散型变量之间的关系，通常用于频数统计。Pandas提供了`pd.crosstab()`函数来生成交叉表。

In [141]:
df

,菜品,颜色,价格,数量
0,白菜,绿,64,88
1,白菜,红,24,98
2,冬瓜,红,0,43
3,辣椒,红,59,51
4,西红柿,白,29,29
5,白菜,白,5,37
6,冬瓜,红,41,36
7,冬瓜,白,43,24
8,冬瓜,白,88,8
9,辣椒,红,61,28


In [ ]:
pd.crosstab(index=df['菜品'],  
            columns=df['颜色'])  # 生成交叉表，统计菜品和颜色的频数

颜色,白,红,绿
菜品,,,
冬瓜,3,2,1
白菜,2,1,2
西红柿,2,2,0
辣椒,0,4,1


## 透视表

透视表是对数据进行汇总和分析的一种工具，可以将数据按照指定的行和列进行分组，并对每个分组进行聚合计算。Pandas提供了`pd.pivot_table()`函数来生成透视表。

In [146]:
# data 数据源 要进行透视表的DataFrame对象
# index指的是透视表的行索引，columns指的是透视表的列索引，values指的是透视表中需要聚合的数值列，aggfunc指的是聚合函数，可以是sum、mean、count等。
pd.pivot_table(data=df, index='菜品', columns='颜色', values='价格', aggfunc='mean')  # 生成透视表，统计菜品和颜色的平均单价

颜色,白,红,绿
菜品,,,
冬瓜,52.666667,20.5,18.0
白菜,10.500000,24.0,48.0
西红柿,25.500000,23.5,NaN
辣椒,NaN,58.0,47.0
